### Create UHVDB by:
1. Identifying unique seqs across UHGV + newly mined viruses
2. Dereplicating at 99.5% ANI and 100% AF

In [ ]:
%%bash
## identify UHGV HQ plus uncertain
# wget https://portal.nersc.gov/cfs/m342/UHGV/genome_catalogs/uhgv_full.fna.gz

# seqkit grep \
#     uhgv_full.fna.gz \
#     --pattern-file uhgv_hq_plus_uncertain.tsv \
#     --out-file uhgv_hq_plus_uncertain.fna.gz

## combine uncertain with confident HQ plus genomes
# wget https://portal.nersc.gov/cfs/m342/UHGV/genome_catalogs/uhgv_hq_plus.fna.gz

# cat uhgv_hq_plus.fna.gz \
#     uhgv_hq_plus_uncertain.fna.gz \
#     > uhgv_hq_plus_confident_w_uncertain.fna.gz

In [ ]:
%%bash
# # run seqhasher to dereplicate genomes
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhgdb/bin/seq-hasher \
#     uhvdb_hq_plus.fna.gz \
#     --multi-kmer-hashing \
#     --circular-kmers \
#     > uhgv.seq-hasher.tsv

In [ ]:
%%bash
# # combine all newly mined HQ viruses
# touch hq_viruses_cat.fasta.gz

# for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/r2025_0*/*/uhvdb/hqfilter/*.hq_viruses.fna.gz; do
#     cat $file >> hq_viruses_cat.fasta.gz
# done

In [ ]:
%%bash
# # trim DTRs on new viruses
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/tr-trimmer \
#     hq_viruses_cat.fasta.gz \
#     --min-length 20 --include-tr-info \
#     > hq_viruses.tr-trimmer.fna

# # dereplicate new viruses
# /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/bin/seq-hasher \
#     hq_viruses.tr-trimmer.fna \
#     --multi-kmer-hashing \
#     --circular-kmers \
#     > hq_viruses.seq-hasher.tsv

In [ ]:
%%bash
# # combine UHGV and new virus seq-hasher outputs to identify unique hashes
# cat uhgv.seq-hasher.tsv \
#     hq_viruses.seq-hasher.tsv \
#     > combined.seq-hasher.tsv

# csvtk uniq \
#     combined.seq-hasher.tsv \
#     --no-header-row \
#     --fields 2 \
#     --tabs \
#     --out-file hq_viruses.csvtk_uniq.tsv.gz

# # extract seq ids
# csvtk cut \
#     --tabs \
#     --fields 1 \
#     hq_viruses.csvtk_uniq.tsv.gz \
#     --out-file hq_viruses.unique_seq_ids.tsv

# # extract unique sequences from UHGV
# seqkit \
#     grep \
#     --pattern-file hq_viruses.unique_seq_ids.tsv \
#     uhgv_hq_plus_confident_w_uncertain.fna.gz \
#     -o uhgv.seqkit_derep.fna.gz

# # extract unique sequences from new viruses
# seqkit \
#     grep \
#     --pattern-file hq_viruses.unique_seq_ids.tsv \
#     hq_viruses.tr-trimmer.fna \
#     -o hq_viruses.seqkit_derep.fna.gz

# combine dereplicated UHGV and new viruses to create UHVDB
# cat uhgv.seqkit_derep.fna.gz \
#     hq_viruses.seqkit_derep.fna.gz \
#     > uhvdb_unique.fna.gz

In [ ]:
%%bash
# count the number of unique HQ sequences
cat hq_viruses.unique_seq_ids.tsv | wc -l
# 591,860 sequences with unique hashes

591860


In [ ]:
%%bash
# # dereplicate UHVDB at 100% ANI and 100% AF shorter sequence
# vclust \
#     prefilter \
#     --in uhvdb_unique.fna.gz \
#     --out uhvdb.vclust_derep_prefilter.txt \
#     --threads 36 \
#     --min-ident 0.99

# vclust \
#     align \
#     --in uhvdb_unique.fna.gz \
#     --out uhvdb.vclust_derep_ani100_qcov100_ani.tsv \
#     --filter uhvdb.vclust_derep_prefilter.txt \
#     --filter-threshold 1.0 \
#     --threads 36 \
#     --out-ani 1.0 \
#     --out-qcov 1.0

# vclust \
#     cluster \
#     --in uhvdb.vclust_derep_ani100_qcov100_ani.tsv \
#     --ids uhvdb.vclust_derep_ani100_qcov100_ani.ids.tsv \
#     --out uhvdb.vclust_derep_ani100_qcov100_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 1.0 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     uhvdb.vclust_derep_ani100_qcov100_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file uhvdb.vclust_derep_ani100_qcov100_reps.tsv

# seqkit \
#     grep \
#     uhvdb_unique.fna.gz \
#     --pattern-file uhvdb.vclust_derep_ani100_qcov100_reps.tsv \
#     --out-file uhvdb.vclust_derep_ani100_qcov100_reps.fna.gz

# 553,951 dereplicated sequences

In [ ]:
%%bash
# # cluster into genomovars at 99.5% ANI and 100% AF shorter sequence
vclust \
    align \
    --in uhvdb_unique.fna.gz \
    --out uhvdb.vclust_genomovars_ani.tsv \
    --filter uhvdb.vclust_derep_prefilter.txt \
    --filter-threshold 0.99 \
    --threads 36 \
    --out-ani 0.995 \
    --out-qcov 1.0

In [3]:
# filter ani output to only contain derep representatives
import polars as pl

derep_reps = set(pl.read_csv("uhvdb.vclust_derep_ani100_qcov100_reps.tsv")['cluster'])

(
    pl.read_csv("uhvdb.vclust_genomovars_ani.tsv", separator="\t")
        .filter(
            (pl.col("query").is_in(derep_reps)) &
            (pl.col("reference").is_in(derep_reps))
        )
        .write_csv("uhvdb.vclust_genomovars_ani_filtered.tsv", separator="\t")
)

In [ ]:
%%bash
### cluster into genomovars at 99.5% ANI and 100% AF shorter sequence ###
# vclust \
#     cluster \
#     --in uhvdb.vclust_genomovars_ani.tsv \
#     --ids uhvdb.vclust_genomovars_ani.ids.tsv \
#     --out uhvdb.vclust_genomovars_clusters.tsv \
#     --algorithm cd-hit \
#     --metric ani \
#     --ani 0.995 \
#     --qcov 1.0 \
#     --out-repr

# csvtk \
#     cut \
#     uhvdb.vclust_genomovars_clusters.tsv \
#     --tabs \
#     --fields cluster | \
# csvtk \
#     uniq \
#     --tabs \
#     --out-file uhvdb.vclust_genomovars_reps.tsv

# seqkit \
#     grep \
#     uhvdb_unique.fna.gz \
#     --pattern-file uhvdb.vclust_genomovars_reps.tsv \
#     --out-file uhvdb.vclust_genomovars_reps.fna.gz

In [5]:
# check that genomovars contains the same sequences as derep
derep_reps = pl.read_csv('uhvdb.vclust_derep_reps.tsv')
genomvars_reps = pl.read_csv('uhvdb.vclust_genomovars_reps.tsv')
print(derep_reps.shape[0], genomvars_reps.shape[0])

genomvars_reps.filter(pl.col('cluster').is_in(derep_reps['cluster'])).shape[0]

458321 458321


458321

In [ ]:
%%bash
# # combine all mine reports into one file
# echo -e "seq_name\ttopology\tcoordinates\tn_genes\tgenetic_code\tvirus_score\tfdr\tn_hallmarks\tmarker_enrichment\ttaxonomy\tgenome\tplasmid_hallmarks\tconj_genes\tzot_genes\tinoviridae_marker_genes\ttotal_markers\tn_dup_markers\tmarker_duplicity\tcontig_length\tprovirus\tproviral_length\tviral_genes\thost_genes\tcompleteness\tcompleteness_method\tkmer_freq\twarningsPrediction\tScore\tPfam\thits\tictv_family\tgenomad_virus_score_95\tviralverify_virus_score_15\tvirus_hallmarks_gt_3\tictv_known_family\tzot_gene\tinoviridae_marker_genes_gt_5\tviralverify_chromosome_plasmid\tcheckv_host_genes_gt_2\tplasmid_hallmarks_gt_0\tconj_genes_gt_0\tmarker_duplicity_penalty\tviral_score_sum\tuhvdb_virus_classification\tcompleteness_2\tcompleteness_method_2\tcheckv_quality\twarnings_2\taai_expected_length\taai_num_hits\taai_top_hit\taai_id\taai_af\tsource_db" \
#     > viruses.csvtk_concat.tsv

# for file in /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/*/*/uhvdb/minereport/*.mine_summary.tsv.gz; do
#     zcat "$file" | tail -n +2 >> viruses.csvtk_concat.tsv
# done

# identify all fasta IDs
zcat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/r2025_0*/*/uhvdb/virusfilter/*.uhvdb_viruses.fna.gz \
    | grep "^>" > uhvdb_virusfilter_ids.txt

In [ ]:
### Load minereport and filter to unique IDs above to get final count of unique seqs
import polars as pl

# read sequence ids
uhvdb_hq_ids = pl.read_csv('uhvdb_virusfilter_ids.txt', has_header=False, new_columns=['seq_header'])

# read mine report
mine_report = pl.read_csv('viruses.csvtk_concat.tsv', separator='\t')

In [ ]:
# dereplicating at 100% rcov and 100% qcov would lead to 549,635 clusters (instead of 458,321)
# dereplicating at 100% ANI, 100% qcov, and 100% rcov would lead to 583,671 clusters